# GARCH en derivados: NVDA, AMD y XOM
**Complemento a árboles binomiales** · Hull, *Options, Futures and Other Derivatives*, 11.ª ed., cap. 13, 22 y 23

**Equipo 6** · Mariann Antonelli Morales Tobias · Eduardo Ayala Saldaña · Ulises Salinas López

EGADE Business School · Maestría en Finanzas, Grupo 40 · Derivados, Dr. Raymundo Díaz Robles

## ¿Qué responde este documento?
1. **¿Qué tan agitada está cada acción hoy** frente a su nivel normal? (volatilidad hoy vs. largo plazo)
2. **¿Qué volatilidad esperamos hasta el 18-dic-2026**, que es cuando vencen nuestras opciones?
3. **¿Esa volatilidad es mayor o menor que la que cobra el mercado?** Si es mayor, la opción está barata; si es menor, cara.
4. **¿Qué rango de precios es razonable** para cada acción al vencimiento y qué tan probable es ganar con la opción que compramos?

| Sección | Tema (Hull) | Qué hacemos |
|---|---|---|
| 1 | Datos | Precios diarios y rendimientos de NVDA, AMD y XOM |
| 2 | 23.1 Volatilidad histórica | Qué tan agitadas estuvieron en los últimos 30 días y en el último año |
| 3 | 23.2 EWMA | Versión que da más peso a lo reciente |
| 4 | 23.3 y 23.5 GARCH(1,1) | Estimar ω, α, β y ver a qué nivel regresa la volatilidad |
| 5 | 23.4 Elegir modelo | Comparar modelos (AIC, BIC, Ljung-Box) |
| 6 | 23.6 Pronóstico | Volatilidad promedio hasta el vencimiento: **la σ que entra al árbol** |
| 7 | 23.7 Correlaciones | Qué tanto se mueven juntas las tres acciones |
| 8 | 22.6 Monte Carlo | Abanico de precios hasta el 18-dic |
| 9 | 22.8 Back testing | ¿El VaR de GARCH mide bien el riesgo? (prueba de Kupiec) |
| 10 | Cap. 13 Árbol binomial | Valorar la call y la put con σ de GARCH vs. σ implícita, y griegas |
| 11 | Cap. 13 Ejercicio anticipado | ¿Cuánto vale poder ejercer antes? |
| 12 | Verificación | Monte Carlo GARCH vs. árbol, y validación contra la presentación |
| 13 | Estrategia | Señal de volatilidad, qué tan sólida es, straddle y resultado esperado de **nuestra** posición, y decisión para Investopedia |
| 14 | Resumen | Conclusiones |
| 15 | Dashboard | Archivo interactivo con todo lo anterior |

**Cómo leerlo:** después de cada tabla o gráfica hay un cuadro **«En palabras simples»** con la conclusión en una o dos frases.
Material educativo; no es una recomendación de inversión.

In [ ]:
# ---- Librerías ----
import os, warnings, json
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.stats import norm, chi2
from IPython.display import display, Markdown

## 0. Parámetros (aquí se cambia todo)
Las cotizaciones de opciones son las capturadas en TradingView (cierre del **30-oct-2026**, capturadas el 1-nov).
Si se actualizan las cotizaciones, basta con cambiar `FECHA_COTIZACION` y la tabla `MERCADO`.

In [ ]:
TICKERS = ["NVDA", "AMD", "XOM"]
NOMBRE = {"NVDA": "NVIDIA", "AMD": "AMD", "XOM": "Exxon Mobil"}
SECTOR = {"NVDA": "Semiconductores (IA)", "AMD": "Semiconductores", "XOM": "Energía (petróleo)"}

FECHA_COTIZACION = "2026-10-30"      # cierre de las cotizaciones; los datos históricos se cortan en esta fecha
VENCIMIENTO = "2026-12-18"           # vencimiento de las opciones
FERIADOS = ["2026-11-26"]            # Thanksgiving (mercado cerrado)
R_FREE = 0.0394                      # tasa libre de riesgo continua (misma de las tareas anteriores)
YEARS = 5                            # años de historia
N_SIM = 100000                         # trayectorias de Monte Carlo
LAMBDA_EWMA = 0.94                   # valor clásico de RiskMetrics
MU_ANUAL = 0.08                      # rendimiento esperado anual SUPUESTO para pronosticar el precio
UMBRAL_SENAL = 0.10                  # 10 % de diferencia entre GARCH y volatilidad implícita
SEED = 123
CARPETA_DATOS = "datos"              # opcional: NVDA.csv, AMD.csv y XOM.csv (Yahoo Finance o TradingView)
USAR_LSEG = True   # True = traer los precios de LSEG (en CodeBook no pide credenciales; fuera de Workspace pide credenciales de Platform)
RICS = {"NVDA": "NVDA.O", "AMD": "AMD.O", "XOM": "XOM.N"}   # códigos de LSEG (RIC) de cada acción
PERMITIR_SIMULADOS = os.environ.get("GARCH_PERMITIR_SIM") == "1"   # False = si no hay datos reales, se detiene (recomendado)
MULT = 100                           # acciones por contrato

# Datos de mercado (TradingView, cierre del 30-oct-2026): spot, strike, dividendo, IV, (bid, ask) y posición comprada
MERCADO = {
    "NVDA": dict(S=204.64, K=200.0, q=0.0047, iv=0.4145, call=(15.55, 15.75), put=(9.85, 10.00),
                 compra="call", contratos=2, reporte="≈ 24-nov"),
    "AMD":  dict(S=504.80, K=500.0, q=0.0,    iv=0.5864, call=(46.85, 48.50), put=(39.70, 40.75),
                 compra="put",  contratos=1, reporte="≈ 3-nov"),
    "XOM":  dict(S=187.25, K=185.0, q=0.0225, iv=0.2991, call=(8.75, 9.20), put=(6.80, 7.30),
                 compra="call", contratos=3, reporte="30-oct (ya ocurrió)"),
}

F0 = pd.Timestamp(FECHA_COTIZACION)
FV = pd.Timestamp(VENCIMIENTO)
DIAS_CAL = (FV - F0).days
T_ANIOS = DIAS_CAL / 365
H = int(np.busday_count((F0 + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
                        (FV + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), holidays=FERIADOS))
COLORES = {"NVDA": "#2a78d6", "AMD": "#eb6834", "XOM": "#1baf7a"}   # paleta validada (azul, naranja, verde agua)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
print(f"Cotización: {FECHA_COTIZACION} | Vencimiento: {VENCIMIENTO} | {DIAS_CAL} días naturales (T = {T_ANIOS:.4f} años) | {H} días hábiles")

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titleweight": "bold", "axes.titlesize": 11, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "legend.frameon": False,
    "legend.fontsize": 8, "font.size": 9,
})
def _fmt(a, b):
    d = dict(a); d.update(b); return d

PCT = FuncFormatter(lambda y, _: f"{y:.0%}")
LECTURAS = {}      # textos «En palabras simples», se reutilizan en el dashboard
TABLAS = {}        # tablas, se reutilizan en el dashboard

def pc(x, d=1):
    return f"{x * 100:.{d}f} %"

def leer(lineas, clave=None, titulo="En palabras simples"):
    """Muestra un cuadro con la interpretación en lenguaje sencillo."""
    if isinstance(lineas, str):
        lineas = [lineas]
    if clave:
        LECTURAS[clave] = lineas
    display(Markdown(f"> **{titulo}**\n>\n" + "\n".join(f"> - {l}" for l in lineas)))

def tabla(df, formato=None, clave=None):
    if clave:
        TABLAS[clave] = (df, formato or {})
    try:
        sty = df.style.format(formato or {})
        try:
            sty = sty.hide(axis="index")
        except Exception:
            sty = sty.hide_index()
        display(sty)
    except Exception:
        display(df)

def mostrar(nombre=None):
    plt.tight_layout()
    plt.show()

## 1. Datos: precios y rendimientos logarítmicos
Se usa $u_i = 100\ln(S_i/S_{i-1})$ (en %). Los modelos de volatilidad trabajan con estos rendimientos diarios.
**Fuente de datos (en este orden):** archivos CSV en la carpeta `datos/` (Yahoo Finance o TradingView) → Yahoo Finance con `yfinance`
→ se detiene con un aviso (los datos simulados solo se usan si `PERMITIR_SIMULADOS = True`, y el documento lo avisa en rojo).

In [ ]:
def leer_csv(path):
    df = pd.read_csv(path)
    cols = {c.lower().strip(): c for c in df.columns}
    fcol = next((cols[k] for k in ("date", "time", "datetime", "fecha") if k in cols), df.columns[0])
    pcol = next((cols[k] for k in ("adj close", "adj_close", "adjclose", "close", "cierre") if k in cols), None)
    if pcol is None:
        raise ValueError(f"{path}: no encontré columna de precio de cierre")
    f = df[fcol]
    fechas = pd.to_datetime(f, unit="s") if pd.api.types.is_numeric_dtype(f) else pd.to_datetime(f, utc=True, errors="coerce").dt.tz_convert(None)
    px = pd.to_numeric(df[pcol].astype(str).str.replace(",", "").str.replace("$", ""), errors="coerce")
    s = pd.Series(px.values, index=fechas.dt.normalize()).dropna()
    return s[~s.index.duplicated(keep="last")].sort_index()


def descargar_lseg():
    """Precios de cierre desde LSEG (librería lseg.data).
    1) Sesión de Workspace / CodeBook (no pide credenciales). 2) Si falla, sesión Platform con App Key, usuario y contraseña
    (se piden al ejecutar y NO se guardan)."""
    import getpass
    import lseg.data as ld
    try:
        ld.open_session()                                     # CodeBook / Workspace Desktop
        print("Sesión LSEG abierta (Workspace / CodeBook)")
    except Exception as e:
        print("No se pudo abrir la sesión de Workspace (", str(e)[:80], "). Se intenta con credenciales de Platform.")
        app_key = os.environ.get("LSEG_APP_KEY") or getpass.getpass("LSEG App Key: ")
        usuario = os.environ.get("LSEG_USER") or input("Usuario LSEG: ")
        clave = os.environ.get("LSEG_PASSWORD") or getpass.getpass("Contraseña LSEG: ")
        sesion = ld.session.platform.Definition(
            app_key=app_key, grant=ld.session.platform.GrantPassword(username=usuario, password=clave), signon_control=True).get_session()
        sesion.open(); ld.session.set_default(sesion)
        print("Sesión LSEG (Platform):", sesion.open_state)
    ini = (F0 - pd.DateOffset(years=YEARS)).strftime("%Y-%m-%d"); fin = (F0 + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = None
    for campo in ("TRDPRC_1", "CLOSE"):
        try:
            df = ld.get_history(universe=list(RICS.values()), fields=[campo], interval="daily", start=ini, end=fin)
            if df is not None and len(df) > 300:
                break
        except Exception as e:
            print(f"LSEG: el campo {campo} no funcionó ({str(e)[:80]})")
    ld.close_session()
    if df is None or len(df) <= 300:
        return None, None
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    df = df.rename(columns={v: k for k, v in RICS.items()}).dropna()
    return df[TICKERS], "LSEG (lseg.data)"


def descargar_precios():
    # 0) LSEG, si se pidió (USAR_LSEG = True)
    if USAR_LSEG:
        try:
            df, fte = descargar_lseg()
            if df is not None:
                return df, fte
        except Exception as e:
            print("No se pudo usar LSEG:", str(e)[:120])
    # 1) CSV locales
    rutas = {t: os.path.join(CARPETA_DATOS, f"{t}.csv") for t in TICKERS}
    if all(os.path.exists(r) for r in rutas.values()):
        df = pd.concat({t: leer_csv(r) for t, r in rutas.items()}, axis=1).dropna()
        return df, "Archivos CSV (Yahoo Finance / TradingView)"
    # 2) Yahoo Finance
    try:
        import yfinance as yf
        ini = F0 - pd.DateOffset(years=YEARS)
        df = yf.download(TICKERS, start=ini, end=F0 + pd.Timedelta(days=1), auto_adjust=True, progress=False, timeout=20)["Close"].dropna()
        if df.shape[0] > 300 and all(t in df.columns for t in TICKERS):
            return df[TICKERS], "Yahoo Finance (yfinance)"
    except Exception as e:
        print("No se pudo descargar de Yahoo Finance:", str(e)[:80])
    return None, None


def datos_sinteticos(n=1260):
    """Serie simulada con GARCH(1,1) (solo respaldo para probar el código)."""
    rng = np.random.default_rng(SEED)
    z = rng.standard_normal((n, 3))
    z[:, 1] = 0.75 * z[:, 0] + np.sqrt(1 - 0.75 ** 2) * z[:, 1]
    par = {"NVDA": (0.235, 0.08, 0.90), "AMD": (0.26, 0.07, 0.91), "XOM": (0.078, 0.07, 0.90)}
    cols = {}
    for j, t in enumerate(TICKERS):
        omega, alpha, beta = par[t]
        v = omega / (1 - alpha - beta)
        u = np.empty(n)
        for i in range(n):
            u[i] = np.sqrt(v) * z[i, j]
            v = omega + alpha * u[i] ** 2 + beta * v
        s = np.exp(np.cumsum(u / 100 + 0.0004))
        cols[t] = s / s[-1] * MERCADO[t]["S"]
    return pd.DataFrame(cols, index=pd.bdate_range(end=F0, periods=n))


precios, fuente = descargar_precios()
SINTETICO = precios is None
if SINTETICO and not PERMITIR_SIMULADOS:
    raise RuntimeError("No se pudieron obtener precios reales. Ejecuta este notebook en Google Colab (descarga con yfinance) "
                       "o coloca NVDA.csv, AMD.csv y XOM.csv en la carpeta 'datos/'. Para probar con datos simulados: PERMITIR_SIMULADOS = True.")
if SINTETICO:
    precios = datos_sinteticos()
    fuente = "DATOS SIMULADOS (no son precios reales de mercado)"
precios = precios[(precios.index <= F0) & (precios.index >= F0 - pd.DateOffset(years=YEARS))]
try:
    precios.to_csv("precios_usados.csv")      # copia de los precios usados (permite reproducir el análisis)
except Exception:
    pass
rend = 100 * np.log(precios / precios.shift(1)).dropna()      # rendimientos en %
S0 = {t: MERCADO[t]["S"] for t in TICKERS}                    # spot de la cotización (el mismo del trabajo de árboles)

if SINTETICO:
    display(Markdown("<div style='background:#fde8e8;border:2px solid #c0392b;padding:10px;border-radius:6px'>"
                     "<b>ATENCIÓN: datos simulados.</b> No son precios reales; solo sirven para probar el código. "
                     "Coloca NVDA.csv, AMD.csv y XOM.csv en la carpeta <code>datos/</code> y vuelve a ejecutar.</div>"))
print("Fuente de datos:", fuente)
print(f"Observaciones: {len(precios)}  |  {precios.index[0].date()} → {precios.index[-1].date()}")
chk = pd.DataFrame({"Último cierre en los datos": precios.iloc[-1], "Spot de cotización (TradingView)": pd.Series(S0)})
chk["Diferencia %"] = 100 * (chk.iloc[:, 0] / chk.iloc[:, 1] - 1)
tabla(chk.reset_index().rename(columns={"index": "Acción"}),
      {"Último cierre en los datos": "{:,.2f}", "Spot de cotización (TradingView)": "{:,.2f}", "Diferencia %": "{:+.2f}"})
if not SINTETICO and (chk["Diferencia %"].abs() > 2).any():
    print("AVISO: el último cierre difiere más de 2 % del spot de la cotización; revisa que el CSV llegue al 30-oct-2026.")

fig, ax = plt.subplots(figsize=(9, 3.6))
base = 100 * precios / precios.iloc[0]
for t in TICKERS:
    ax.plot(base.index, base[t], color=COLORES[t], lw=1.6, label=t)
    ax.annotate(t, (base.index[-1], base[t].iloc[-1]), xytext=(4, 0), textcoords="offset points", va="center", color=INK, fontsize=9)
ax.set_title("Precio de cada acción (base 100 al inicio de la muestra)")
ax.legend(loc="upper left"); ax.margins(x=0.06)
mostrar()
leer([f"Los precios van de {precios.index[0].date()} a {precios.index[-1].date()} ({len(rend)} rendimientos diarios). "
      "Con la base en 100 se ve cuál acción creció más y cuál fue más accidentada.",
      f"Fuente: {fuente}."], "datos")

## 2. Volatilidad histórica (Hull 23.1)
Es la desviación estándar de los rendimientos recientes, anualizada con $\sqrt{252}$. Su debilidad: da el mismo peso a todos los días
de la ventana, así que reacciona lento a un shock.

In [ ]:
def vol_anual_desde_var(v):
    """Varianza diaria en %² → volatilidad anual (decimal)."""
    return np.sqrt(252 * np.asarray(v)) / 100

vol_30 = rend.rolling(30).std() * np.sqrt(252) / 100
vol_hist_1a = rend.iloc[-252:].std() * np.sqrt(252) / 100
vol_30_hoy = vol_30.iloc[-1]

hist = pd.DataFrame({"Acción": TICKERS,
                     "Últimos 30 días": [vol_30_hoy[t] for t in TICKERS],
                     "Último año": [vol_hist_1a[t] for t in TICKERS]})
hist["¿Más agitada que su año?"] = np.where(hist["Últimos 30 días"] > hist["Último año"], "Sí", "No")
tabla(hist, {"Últimos 30 días": "{:.1%}", "Último año": "{:.1%}"}, "hist")
leer([f"**{t}**: los últimos 30 días se movió con una volatilidad de {pc(vol_30_hoy[t])} anual, contra {pc(vol_hist_1a[t])} del último año. "
      + ("Está más agitada que de costumbre." if vol_30_hoy[t] > vol_hist_1a[t] else "Está más tranquila que de costumbre.")
      for t in TICKERS] +
     ["Una volatilidad de 40 % anual significa que, en un año típico, el precio puede subir o bajar alrededor de 40 % (una desviación estándar)."], "vol_hist")

## 3. EWMA (Hull 23.2)
$\sigma_n^2=\lambda\sigma_{n-1}^2+(1-\lambda)u_{n-1}^2$ — pesos que bajan exponencialmente. Con $\lambda=0.94$ casi todo el peso está en
las últimas semanas. Es un caso especial de GARCH con $\omega=0$, $\alpha=1-\lambda$ y $\beta=\lambda$.

In [ ]:
def var_garch(params, u, v0=None):
    """Serie de varianzas condicionales de GARCH(1,1). v[t] usa u[t-1] y v[t-1]."""
    omega, alpha, beta = params
    if v0 is None:
        v0 = np.var(u)
    x = omega + alpha * u[:-1] ** 2
    y, _ = lfilter([1.0], [1.0, -beta], x, zi=[beta * v0])
    return np.r_[v0, y]

def var_ewma(u, lam=LAMBDA_EWMA):
    return var_garch((0.0, 1 - lam, lam), u, v0=np.var(u[:30]))

def loglik(v, u):
    """Log-verosimilitud gaussiana (Hull 23.5: maximizar Σ[-ln v − u²/v])."""
    return -0.5 * np.sum(np.log(2 * np.pi) + np.log(v) + u ** 2 / v)

ewma_hoy = {}
for t in TICKERS:
    u = rend[t].values
    ewma_hoy[t] = float(vol_anual_desde_var(lam_var := (LAMBDA_EWMA * var_ewma(u)[-1] + (1 - LAMBDA_EWMA) * u[-1] ** 2)))
leer([f"**{t}**: con EWMA (λ = 0.94) la volatilidad de hoy es {pc(ewma_hoy[t])}." for t in TICKERS] +
     ["EWMA reacciona rápido a un día fuerte, pero no tiene un nivel «normal» al que regresar; por eso para pronosticar a 2 meses usamos GARCH."], "ewma")

## 4. GARCH(1,1) por máxima verosimilitud (Hull 23.3 y 23.5)
$$\sigma_n^2=\omega+\alpha u_{n-1}^2+\beta\sigma_{n-1}^2,\qquad V_L=\frac{\omega}{1-\alpha-\beta}$$
- **α**: cuánto pesa el shock de ayer. **β**: cuánta memoria tiene la varianza.
- **α + β** (persistencia) debe ser menor que 1; cerca de 1 los shocks tardan mucho en desaparecer.
- **V_L**: varianza de largo plazo hacia la que regresa el modelo.
- Los parámetros se eligen para que los rendimientos observados sean lo más probables posible (máxima verosimilitud).

In [ ]:
def ajustar_garch(u):
    v = np.var(u)
    mejor = None
    for a0, b0 in [(0.08, 0.90), (0.05, 0.93), (0.12, 0.80)]:       # varios puntos de partida por robustez
        x0 = [v * (1 - a0 - b0), a0, b0]
        res = minimize(lambda p: -loglik(var_garch(p, u), u), x0, method="SLSQP",
                       bounds=[(1e-6, None), (1e-4, 0.5), (0.30, 0.9998)],
                       constraints=({"type": "ineq", "fun": lambda p: 0.9999 - p[1] - p[2]},),
                       options={"maxiter": 500, "ftol": 1e-10})
        if mejor is None or res.fun < mejor.fun:
            mejor = res
    return mejor.x

M = {}
for t in TICKERS:
    u = rend[t].values
    p = ajustar_garch(u)
    v = var_garch(p, u)
    omega, alpha, beta = p
    VL = omega / (1 - alpha - beta)
    M[t] = dict(u=u, p=p, v=v, VL=VL, v_next=omega + alpha * u[-1] ** 2 + beta * v[-1])

par = pd.DataFrame({
    t: {"ω (omega)": M[t]["p"][0], "α (shock de ayer)": M[t]["p"][1], "β (memoria)": M[t]["p"][2],
        "α + β (persistencia)": M[t]["p"][1] + M[t]["p"][2],
        "Vol. largo plazo (anual)": vol_anual_desde_var(M[t]["VL"]),
        "Vol. hoy según GARCH (anual)": vol_anual_desde_var(M[t]["v_next"]),
        "Vida media del shock (días)": np.log(0.5) / np.log(M[t]["p"][1] + M[t]["p"][2])}
    for t in TICKERS}).T.reset_index().rename(columns={"index": "Acción"})
tabla(par, {"ω (omega)": "{:.4f}", "α (shock de ayer)": "{:.4f}", "β (memoria)": "{:.4f}", "α + β (persistencia)": "{:.4f}",
            "Vol. largo plazo (anual)": "{:.1%}", "Vol. hoy según GARCH (anual)": "{:.1%}", "Vida media del shock (días)": "{:.0f}"}, "garch")

# Validación contra el paquete `arch`
val = []
try:
    from arch import arch_model
    for t in TICKERS:
        am = arch_model(M[t]["u"], mean="Zero", vol="GARCH", p=1, q=1, dist="normal").fit(disp="off")
        val.append({"Acción": t, "α paquete arch": am.params["alpha[1]"], "α nuestro": M[t]["p"][1],
                    "β paquete arch": am.params["beta[1]"], "β nuestro": M[t]["p"][2],
                    "Log-verosim. arch": am.loglikelihood, "Log-verosim. nuestro": loglik(M[t]["v"], M[t]["u"])})
    tabla(pd.DataFrame(val), _fmt({c: "{:.4f}" for c in ["α paquete arch", "α nuestro", "β paquete arch", "β nuestro"]},
          {"Log-verosim. arch": "{:.2f}", "Log-verosim. nuestro": "{:.2f}"}))
    print("Validación: nuestro código de máxima verosimilitud y el paquete `arch` deben dar valores muy parecidos.")
except Exception as e:
    print("Validación con `arch` omitida:", e)

lin = []
for t in TICKERS:
    a, b = M[t]["p"][1], M[t]["p"][2]
    vh, vl = vol_anual_desde_var(M[t]["v_next"]), vol_anual_desde_var(M[t]["VL"])
    lin.append(f"**{t}**: persistencia {a + b:.3f}; un shock pierde la mitad de su efecto en ~{np.log(0.5) / np.log(a + b):.0f} días. "
               f"Su volatilidad «normal» es {pc(vl)} y hoy el modelo ve {pc(vh)}: "
               + ("está **por encima** de lo normal, así que se espera que baje." if vh > vl * 1.02 else
                  "está **por debajo** de lo normal, así que se espera que suba." if vh < vl * 0.98 else "está en su nivel normal."))
leer(lin + ["α chico y β grande (por ejemplo 0.08 y 0.90) es lo típico en acciones: la volatilidad cambia poco a poco, no de golpe."], "garch")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), sharey=False)
for ax, t in zip(axs, TICKERS):
    idx = rend.index
    ax.plot(idx, vol_30[t], color="#9a9890", alpha=0.7, lw=1, label="Histórica 30 días")
    ax.plot(idx, vol_anual_desde_var(var_ewma(M[t]["u"])), color=INK, lw=1, ls="--", label="EWMA λ=0.94")
    ax.plot(idx, vol_anual_desde_var(M[t]["v"]), color=COLORES[t], lw=1.7, label="GARCH(1,1)")
    ax.axhline(vol_anual_desde_var(M[t]["VL"]), color=COLORES[t], ls=":", lw=1.4, label="Largo plazo (V_L)")
    ax.set_title(f"{t}: volatilidad anualizada"); ax.yaxis.set_major_formatter(PCT)
    ax.legend(fontsize=7, loc="upper left")
    for lab in ax.get_xticklabels():
        lab.set_rotation(30)
mostrar()
leer("GARCH (línea de color) sube rápido tras un periodo agitado y luego regresa poco a poco hacia su nivel de largo plazo (línea punteada). "
     "La línea gris, la histórica de 30 días, hace algo parecido pero con rezago.", "graf_vol")

## 5. ¿Qué modelo elegir? (Hull 23.4 y 23.5)
Se compara la **log-verosimilitud** (más alta es mejor), **AIC y BIC** (penalizan parámetros; más bajo es mejor) y la **prueba de
Ljung-Box** sobre $u^2/\sigma^2$: si p > 0.05, el modelo ya capturó el agrupamiento de volatilidad.

In [ ]:
def lb_pvalue(serie, lags=10):
    x = np.asarray(serie, dtype=float); x = x - x.mean(); n = len(x)
    r = np.array([np.sum(x[k:] * x[:-k]) for k in range(1, lags + 1)]) / np.sum(x * x)
    q = n * (n + 2) * np.sum(r ** 2 / (n - np.arange(1, lags + 1)))
    return float(1 - chi2.cdf(q, lags))

filas = []
for t in TICKERS:
    u, n = M[t]["u"], len(M[t]["u"])
    grid = np.arange(0.80, 0.995, 0.005)
    lam_opt = grid[int(np.argmax([loglik(var_ewma(u, l), u) for l in grid]))]
    M[t]["lam_opt"] = lam_opt
    modelos = {"Volatilidad constante": (np.full(n, np.var(u)), 1), "EWMA λ=0.94": (var_ewma(u, 0.94), 0),
               f"EWMA λ óptimo ({lam_opt:.3f})": (var_ewma(u, lam_opt), 1), "GARCH(1,1)": (M[t]["v"], 3)}
    for nombre, (v, k) in modelos.items():
        ll = loglik(v, u)
        filas.append({"Acción": t, "Modelo": nombre, "Log-verosim.": ll, "AIC": 2 * k - 2 * ll,
                      "BIC": k * np.log(n) - 2 * ll, "p Ljung-Box": lb_pvalue(u ** 2 / v)})
comparacion = pd.DataFrame(filas)
tabla(comparacion, {"Log-verosim.": "{:.1f}", "AIC": "{:.1f}", "BIC": "{:.1f}", "p Ljung-Box": "{:.3f}"}, "comparacion")
lb_raw = {t: lb_pvalue(M[t]["u"] ** 2) for t in TICKERS}
print("p de Ljung-Box de u² sin modelar (si es pequeño, hay agrupamiento):", {t: round(v, 4) for t, v in lb_raw.items()})

lin = []
for t in TICKERS:
    c = comparacion[comparacion["Acción"] == t]
    mejor = c.loc[c["BIC"].idxmin(), "Modelo"]
    bic_g = float(c[c["Modelo"] == "GARCH(1,1)"]["BIC"].iloc[0]); dif = bic_g - float(c["BIC"].min())
    pg = float(c[c["Modelo"] == "GARCH(1,1)"]["p Ljung-Box"].iloc[0])
    txt_cl = (f"Sin modelar, la volatilidad sí se agrupa (Ljung-Box p = {lb_raw[t]:.3f})" if lb_raw[t] < 0.05
              else f"Sin modelar, el agrupamiento es dudoso (Ljung-Box p = {lb_raw[t]:.3f}, arriba de 0.05)")
    txt_g = "ya no queda agrupamiento sin explicar" if pg > 0.10 else ("apenas pasa la prueba" if pg > 0.05 else "todavía queda algo de agrupamiento")
    uso = ("GARCH es el mejor modelo según BIC." if mejor == "GARCH(1,1)" else
           f"El mejor según BIC es **{mejor}**; GARCH queda {dif:.1f} puntos por encima ({'una diferencia pequeña' if dif < 6 else 'una diferencia moderada' if dif < 10 else 'una diferencia notable'}), "
           "pero lo usamos porque es el único que regresa a un nivel de largo plazo.")
    lin.append(f"**{t}**: {uso} {txt_cl}; con GARCH el p de Ljung-Box es {pg:.2f} ({txt_g}).")
leer(lin + ["Regla práctica: menor AIC/BIC = mejor ajuste. Cuando EWMA gana suele ser porque la persistencia es casi 1: la reversión a la media de GARCH es muy lenta y su pronóstico a 2 meses es menos confiable. Eso se revisa en la sección 13b."], "modelo")

## 6. Pronóstico de volatilidad con GARCH(1,1) (Hull 23.6)
$$E[\sigma^2_{n+k}]=V_L+(\alpha+\beta)^k(\sigma^2_{n}-V_L)$$
La varianza esperada **regresa** hacia $V_L$. Una opción que vence en $T$ días se valora con la **volatilidad promedio** de esos días
(no la de hoy). **Esa es la σ que entra al árbol binomial.** Aquí $T$ = días hábiles hasta el 18-dic-2026.

In [ ]:
def pron_var(params, u, v, Hd):
    omega, alpha, beta = params
    VL = omega / (1 - alpha - beta)
    v_next = omega + alpha * u[-1] ** 2 + beta * v[-1]
    k = np.arange(Hd)
    return VL + (alpha + beta) ** k * (v_next - VL)

def vol_promedio(t, dias):
    """Volatilidad anual promedio esperada en los próximos `dias` días hábiles."""
    fv = pron_var(M[t]["p"], M[t]["u"], M[t]["v"], max(int(dias), 1))
    return float(np.sqrt(252 * fv.mean()) / 100)

def vol_promedio_hull(t, dias):
    """Fórmula continua de Hull: V_L + (1−e^{−aT})/(aT)·(V(0)−V_L), a = ln(1/(α+β))."""
    omega, alpha, beta = M[t]["p"]
    a = np.log(1 / (alpha + beta))
    return float(np.sqrt(252 * (M[t]["VL"] + (1 - np.exp(-a * dias)) / (a * dias) * (M[t]["v_next"] - M[t]["VL"]))) / 100)

plazos = [10, 30, H, 126, 252]
est = pd.DataFrame({t: [vol_promedio(t, d) for d in plazos] for t in TICKERS},
                   index=[f"{d} días" + (" (vencimiento)" if d == H else "") for d in plazos])
tabla(est.reset_index().rename(columns={"index": "Plazo"}), {t: "{:.1%}" for t in TICKERS}, "estructura")
print("Control con la fórmula continua de Hull (hasta el vencimiento):", {t: f"{vol_promedio_hull(t, H):.1%}" for t in TICKERS})

VOL_GARCH = {t: vol_promedio(t, H) for t in TICKERS}
VOL_HOY = {t: float(vol_anual_desde_var(M[t]["v_next"])) for t in TICKERS}
VOL_LP = {t: float(vol_anual_desde_var(M[t]["VL"])) for t in TICKERS}
IV = {t: MERCADO[t]["iv"] for t in TICKERS}

fig, ax = plt.subplots(figsize=(9, 4))
dd = np.arange(1, 253)
for t in TICKERS:
    fv = pron_var(M[t]["p"], M[t]["u"], M[t]["v"], 252)
    prom = np.sqrt(252 * np.cumsum(fv) / dd) / 100            # volatilidad promedio hasta cada día
    ax.plot(dd, vol_anual_desde_var(fv), color=COLORES[t], lw=1.2, alpha=0.55)
    ax.plot(dd, prom, color=COLORES[t], lw=2.2, label=f"{t}: σ promedio hasta cada día")
    ax.scatter([H], [IV[t]], marker="D", s=46, color=COLORES[t], edgecolor="white", zorder=5)
    ax.annotate(f"{t} · implícita {IV[t]:.0%}", (H, IV[t]), xytext=(8, 0), textcoords="offset points", va="center", fontsize=8, color=INK)
ax.axvline(H, color=INK2, ls="--", lw=1)
ax.annotate("vencimiento\n18-dic", (H, ax.get_ylim()[0]), xytext=(-4, 10), textcoords="offset points", ha="right", fontsize=8, color=INK2)
ax.set_xlabel("Días hábiles hacia adelante"); ax.yaxis.set_major_formatter(PCT)
ax.set_title("Volatilidad pronosticada por GARCH (línea gruesa) vs. volatilidad implícita (rombo)")
ax.legend(loc="best")
mostrar()

lin = []
for t in TICKERS:
    r = VOL_GARCH[t] / IV[t]
    lin.append(f"**{t}**: hasta el vencimiento GARCH espera una volatilidad promedio de **{pc(VOL_GARCH[t])}**; el mercado cobra {pc(IV[t])} "
               f"(razón {r:.2f}). Hoy está en {pc(VOL_HOY[t])} y su nivel normal es {pc(VOL_LP[t])}.")
leer(lin + ["La línea gruesa sube o baja según la acción esté hoy por debajo o por encima de su nivel normal; el rombo es lo que paga hoy el mercado.",
            "**Ojo:** GARCH no conoce las fechas de reporte de resultados y la volatilidad implícita sí las incluye "
            "(AMD ≈ 3-nov y NVDA ≈ 24-nov; XOM ya reportó el 30-oct y su precio ya lo refleja). Es normal que GARCH quede algo por debajo de la implícita cuando hay un reporte dentro de la vida de la opción."], "pronostico")

## 7. Correlación entre las acciones (Hull 23.7)
Con EWMA: $\text{cov}_n=\lambda\,\text{cov}_{n-1}+(1-\lambda)x_{n-1}y_{n-1}$ y $\rho_n=\text{cov}_n/(\sigma_x\sigma_y)$.
Importa para el portafolio: a mayor correlación, menos diversificación.

In [ ]:
PARES = [("NVDA", "AMD"), ("NVDA", "XOM"), ("AMD", "XOM")]
CORR = {}
fig, axs = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, (a, b) in zip(axs, PARES):
    x, y = rend[a].values, rend[b].values
    cov = np.empty(len(x)); cov[0] = np.mean(x[:30] * y[:30])
    for i in range(1, len(x)):
        cov[i] = LAMBDA_EWMA * cov[i - 1] + (1 - LAMBDA_EWMA) * x[i - 1] * y[i - 1]
    c_ew = cov / np.sqrt(var_ewma(x) * var_ewma(y))
    c_60 = rend[a].rolling(60).corr(rend[b])
    CORR[(a, b)] = dict(hoy=float(c_ew[-1]), media=float(np.nanmean(c_ew)), serie=c_ew, c60=c_60)
    ax.plot(rend.index, c_60, color="#9a9890", alpha=0.8, lw=1, label="Móvil 60 días")
    ax.plot(rend.index, c_ew, color=INK, lw=1.5, label="EWMA λ=0.94")
    ax.axhline(0, color=INK2, lw=0.8)
    ax.set_title(f"{a} – {b}"); ax.set_ylim(-0.3, 1)
    for lab in ax.get_xticklabels():
        lab.set_rotation(30)
axs[0].legend(loc="lower left")
mostrar()
cor_df = pd.DataFrame([{"Par": f"{a} – {b}", "Correlación hoy (EWMA)": v["hoy"], "Promedio de la muestra": v["media"]} for (a, b), v in CORR.items()])
tabla(cor_df, {"Correlación hoy (EWMA)": "{:.2f}", "Promedio de la muestra": "{:.2f}"}, "correlacion")
leer([f"**{a} – {b}**: correlación {v['hoy']:.2f} hoy (promedio {v['media']:.2f}). "
      + ("Se mueven casi juntas: tener las dos diversifica poco." if v["hoy"] > 0.6 else
         "Se mueven algo juntas: diversifican a medias." if v["hoy"] > 0.3 else "Se mueven casi por separado: diversifican bien.")
      for (a, b), v in CORR.items()] +
     ["Correlación 1 = se mueven idénticas; 0 = no tienen relación. Si las dos acciones de semiconductores caen juntas, las dos opciones sufren a la vez; XOM (energía) ayuda a repartir el riesgo."], "correlacion")

## 8. Monte Carlo con GARCH: pronóstico del precio (Hull 22.6)
Se simulan miles de trayectorias: cada día $u_t=\sigma_t z_t$ y la varianza se actualiza con GARCH. El resultado es una **distribución**
del precio al 18-dic (no un solo número).
- Para **pronosticar la acción** se usa un rendimiento esperado **supuesto** (`MU_ANUAL`).
- Para **valorar opciones** se usa la tasa libre de riesgo menos dividendos $(r-q)$ y la corrección $-\tfrac12\sigma_t^2$ (Duan, 1995).

In [ ]:
def simular_trayectorias(t, S0_, Hd, n_sim, deriva_anual, rng):
    omega, alpha, beta = M[t]["p"]
    v = np.full(n_sim, omega + alpha * M[t]["u"][-1] ** 2 + beta * M[t]["v"][-1])      # varianza del día n+1, en %²
    S = np.empty((Hd + 1, n_sim)); S[0] = S0_
    for k in range(1, Hd + 1):
        sig = np.sqrt(v)
        z = rng.standard_normal(n_sim)
        S[k] = S[k - 1] * np.exp(deriva_anual / 252 - 0.5 * (sig / 100) ** 2 + (sig / 100) * z)
        v = omega + alpha * (sig * z) ** 2 + beta * v
    return S

rng = np.random.default_rng(SEED)
SIM = {t: simular_trayectorias(t, S0[t], H, N_SIM, MU_ANUAL, rng) for t in TICKERS}
res_mc = {}
for t in TICKERS:
    ST = SIM[t][-1]
    q5, q25, q50, q75, q95 = np.percentile(ST, [5, 25, 50, 75, 95])
    r10 = SIM[t][10] / S0[t] - 1
    res_mc[t] = {"Precio hoy": S0[t], "P5": q5, "P25": q25, "Mediana": q50, "P75": q75, "P95": q95,
                 "Prob. de subir": (ST > S0[t]).mean(), "VaR 99 % a 10 días": -np.percentile(r10, 1)}
mc_df = pd.DataFrame(res_mc).T.reset_index().rename(columns={"index": "Acción"})
tabla(mc_df, _fmt({c: "{:,.2f}" for c in ["Precio hoy", "P5", "P25", "Mediana", "P75", "P95"]}, {"Prob. de subir": "{:.1%}", "VaR 99 % a 10 días": "{:.1%}"}), "mc")

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
for ax, t in zip(axs, TICKERS):
    dias = np.arange(H + 1)
    p = np.percentile(SIM[t], [5, 25, 50, 75, 95], axis=1)
    ax.fill_between(dias, p[0], p[4], color=COLORES[t], alpha=0.16, label="Percentil 5–95")
    ax.fill_between(dias, p[1], p[3], color=COLORES[t], alpha=0.32, label="Percentil 25–75")
    ax.plot(dias, p[2], color=COLORES[t], lw=2, label="Mediana")
    ax.axhline(S0[t], color=INK2, ls=":", lw=1)
    ax.axhline(MERCADO[t]["K"], color=INK, ls="--", lw=1)
    ax.annotate(f"strike {MERCADO[t]['K']:,.0f}", (0, MERCADO[t]["K"]), xytext=(3, 4 if MERCADO[t]["K"] >= S0[t] else -11), textcoords="offset points", fontsize=8, color=INK)
    ax.set_title(f"{t}: precio a {H} días hábiles"); ax.set_xlabel("Días hábiles"); ax.legend(fontsize=7, loc="upper left")
mostrar()
leer([f"**{t}**: hoy vale {S0[t]:,.2f}. Con 90 % de probabilidad terminará el 18-dic entre **{res_mc[t]['P5']:,.0f} y {res_mc[t]['P95']:,.0f}**; "
      f"el caso central (mediana) es {res_mc[t]['Mediana']:,.0f}. La probabilidad de que suba es {pc(res_mc[t]['Prob. de subir'], 0)}."
      for t in TICKERS] +
     ["El abanico se ensancha con el tiempo: cuanto más lejos, más incertidumbre. Un abanico más ancho significa más volatilidad y opciones más caras.",
      f"Aviso: el centro del abanico depende del rendimiento esperado **supuesto** ({MU_ANUAL:.0%} anual). Lo que sí viene de los datos es el ancho."], "mc")

## 9. Back testing del VaR (Hull 22.8)
Se calcula el VaR de 1 día con la volatilidad de cada modelo y se cuentan las **excepciones** (días en que la pérdida fue mayor que el VaR).
Con 99 % se espera ~1 % de excepciones. La **prueba de Kupiec** dice si el número observado es compatible con lo esperado
(p > 0.05: el modelo pasa). GARCH se vuelve a estimar cada 63 días usando solo el pasado (sin mirar el futuro).

In [ ]:
def kupiec(x, N, p):
    lr = -2 * N * np.log(1 - p) if x == 0 else -2 * ((N - x) * np.log((1 - p) / (1 - x / N)) + x * np.log(p / (x / N)))
    return float(1 - chi2.cdf(lr, 1))

def preds_backtest(t, inicio, refit=63):
    u = M[t]["u"]; n = len(u)
    pred = {"GARCH(1,1)": np.full(n, np.nan), "EWMA": var_ewma(u), "Histórica 250d": np.full(n, np.nan)}
    for s in range(inicio, n, refit):
        fin = min(n, s + refit)
        p = ajustar_garch(u[:s])
        pred["GARCH(1,1)"][s:fin] = var_garch(p, u[:fin], v0=np.var(u[:s]))[s:fin]
    for s in range(inicio, n):
        pred["Histórica 250d"][s] = np.var(u[s - 250:s])
    return pred

inicio_bt = int(len(rend) * 0.6)
filas_bt, serie_var = [], {}
for t in TICKERS:
    pred = preds_backtest(t, inicio_bt)
    u = M[t]["u"]
    for nivel in (0.95, 0.99):
        z = norm.ppf(nivel)
        for nombre, vp in pred.items():
            exc = u[inicio_bt:] < -z * np.sqrt(vp[inicio_bt:])
            N, x = len(exc), int(exc.sum())
            filas_bt.append({"Acción": t, "Confianza": f"{nivel:.0%}", "Modelo": nombre, "Días": N, "Excepciones": x,
                             "Tasa observada": x / N, "Tasa esperada": 1 - nivel, "p Kupiec": kupiec(x, N, 1 - nivel)})
    serie_var[t] = (pred["GARCH(1,1)"], norm.ppf(0.99))
bt = pd.DataFrame(filas_bt)
tabla(bt, {"Tasa observada": "{:.2%}", "Tasa esperada": "{:.2%}", "p Kupiec": "{:.3f}"}, "backtest")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, t in zip(axs, TICKERS):
    u = M[t]["u"]; vg, z = serie_var[t]
    ax.plot(rend.index[inicio_bt:], u[inicio_bt:], color="#9a9890", lw=0.7, label="Rendimiento diario (%)")
    ax.plot(rend.index[inicio_bt:], -z * np.sqrt(vg[inicio_bt:]), color=COLORES[t], lw=1.4, label="−VaR 99 % (GARCH)")
    exc = u[inicio_bt:] < -z * np.sqrt(vg[inicio_bt:])
    ax.scatter(rend.index[inicio_bt:][exc], u[inicio_bt:][exc], color=INK, zorder=3, s=16, label="Excepciones")
    ax.set_title(f"{t}: VaR 99 % a 1 día"); ax.legend(fontsize=7, loc="lower left")
    for lab in ax.get_xticklabels():
        lab.set_rotation(30)
mostrar()
def veredicto(f):
    if f["p Kupiec"] > 0.05:
        return "**pasa**"
    return ("**no pasa**: hubo menos excepciones de las esperadas (modelo demasiado cauteloso)" if f["Tasa observada"] < f["Tasa esperada"]
            else "**no pasa**: hubo más excepciones de las esperadas (el modelo subestima el riesgo)")
lin = []
for t in TICKERS:
    g = bt[(bt["Acción"] == t) & (bt["Modelo"] == "GARCH(1,1)")]
    g99 = g[g["Confianza"] == "99%"].iloc[0]; g95 = g[g["Confianza"] == "95%"].iloc[0]
    lin.append(f"**{t}** · 99 %: esperábamos {g99['Tasa esperada']:.0%} de días fuera del VaR y hubo {g99['Tasa observada']:.2%} ({int(g99['Excepciones'])} de {int(g99['Días'])}); Kupiec p = {g99['p Kupiec']:.2f} → {veredicto(g99)}. "
               f"95 %: {g95['Tasa observada']:.2%} vs. {g95['Tasa esperada']:.0%} (p = {g95['p Kupiec']:.2f}) → {veredicto(g95)}.")
leer(lin + ["Cada punto negro es un día en que la pérdida fue mayor que el riesgo pronosticado. Si hay muchos más de los esperados, el modelo es optimista; "
            "si hay muy pocos, es demasiado cauteloso. GARCH con distribución normal suele quedarse corto en las colas extremas."], "backtest")

## 10. Opciones: árbol binomial (cap. 13) con la volatilidad de GARCH
El árbol CRR necesita **una** volatilidad. GARCH entrega la **volatilidad promedio esperada** durante la vida de la opción
y la comparamos con la histórica y con la **implícita** del mercado (TradingView, cierre del 30-oct):
- σ GARCH **mayor** que la implícita → la opción parece **barata**.
- σ GARCH **menor** que la implícita → la opción parece **cara**.

El árbol es el mismo del trabajo anterior (Cox-Ross-Rubinstein, 500 pasos, opción americana, con dividendos): **solo cambia la σ que le damos**.

In [ ]:
def crr(S0_, K, r, sigma, T, N=500, tipo="put", americana=True, q=0.0):
    dt = T / N
    u_ = np.exp(sigma * np.sqrt(dt)); d_ = 1 / u_
    p = (np.exp((r - q) * dt) - d_) / (u_ - d_); disc = np.exp(-r * dt)
    sg = 1 if tipo == "call" else -1
    niveles = [None] * 3
    j = np.arange(N + 1)
    V = np.maximum(sg * (S0_ * u_ ** (N - j) * d_ ** j - K), 0.0)
    if N <= 2: niveles[N] = V
    for i in range(N - 1, -1, -1):
        k = np.arange(i + 1)
        cont = disc * (p * V[:i + 1] + (1 - p) * V[1:i + 2])
        V = np.maximum(cont, np.maximum(sg * (S0_ * u_ ** (i - k) * d_ ** k - K), 0.0)) if americana else cont
        if i <= 2: niveles[i] = V
    return V[0], niveles, u_, d_, dt

def precio(S0_, K, sigma, T, tipo, americana=True, q=0.0, N=500, r=None):
    return crr(S0_, K, R_FREE if r is None else r, sigma, T, N, tipo, americana, q)[0]

def griegas(S0_, K, r, sigma, T, N=500, tipo="put", americana=True, q=0.0):
    pr, niv, u_, d_, dt = crr(S0_, K, r, sigma, T, N, tipo, americana, q)
    f0, f1, f2 = niv[0][0], niv[1], niv[2]
    delta = (f1[0] - f1[1]) / (S0_ * u_ - S0_ * d_)
    gamma = ((f2[0] - f2[1]) / (S0_ * u_ ** 2 - S0_) - (f2[1] - f2[2]) / (S0_ - S0_ * d_ ** 2)) / (0.5 * (S0_ * u_ ** 2 - S0_ * d_ ** 2))
    theta = (f2[1] - f0) / (2 * dt) / 365
    vega = crr(S0_, K, r, sigma + 0.01, T, N, tipo, americana, q)[0] - pr
    rho = crr(S0_, K, r + 0.01, sigma, T, N, tipo, americana, q)[0] - pr
    return dict(precio=pr, delta=delta, gamma=gamma, theta=theta, vega=vega, rho=rho)

OPC = {}
for t in TICKERS:
    m = MERCADO[t]
    OPC[t] = dict(K=m["K"], T=T_ANIOS, q=m["q"], iv=m["iv"], vol_garch=VOL_GARCH[t], vol_hist=float(vol_hist_1a[t]), S0=m["S"],
                  compra=m["compra"], contratos=m["contratos"])

def mid(par):
    return (par[0] + par[1]) / 2

filas = []
for t in TICKERS:
    c, m = OPC[t], MERCADO[t]
    for tipo in ("call", "put"):
        pv = {k: precio(c["S0"], c["K"], c[k], c["T"], tipo, True, c["q"]) for k in ("vol_hist", "vol_garch", "iv")}
        bid, ask = m[tipo]
        filas.append({"Acción": t, "Opción": f"{tipo} {c['K']:,.0f}" + ("  ◄ comprada" if tipo == c["compra"] else ""),
                      "σ histórica 1a": pv["vol_hist"], "σ GARCH": pv["vol_garch"], "σ implícita": pv["iv"],
                      "Mercado (medio)": mid((bid, ask)), "Ask": ask, "GARCH vs. ask %": 100 * (pv["vol_garch"] / ask - 1)})
precios_op = pd.DataFrame(filas)
tabla(pd.DataFrame([{"Acción": t, "σ histórica 1 año": OPC[t]["vol_hist"], "σ GARCH (promedio a vencimiento)": OPC[t]["vol_garch"],
                     "σ implícita (mercado)": OPC[t]["iv"], "Razón GARCH / implícita": OPC[t]["vol_garch"] / OPC[t]["iv"]} for t in TICKERS]),
      {"σ histórica 1 año": "{:.1%}", "σ GARCH (promedio a vencimiento)": "{:.1%}", "σ implícita (mercado)": "{:.1%}", "Razón GARCH / implícita": "{:.2f}"}, "sigmas")
display(Markdown("**Valor de cada opción americana (USD por acción) según la σ que se use en el árbol:**"))
tabla(precios_op, _fmt({c: "{:,.2f}" for c in ["σ histórica 1a", "σ GARCH", "σ implícita", "Mercado (medio)", "Ask"]}, {"GARCH vs. ask %": "{:+.1f}"}), "precios_op")

lin = []
for t in TICKERS:
    c = OPC[t]; f = precios_op[(precios_op["Acción"] == t) & precios_op["Opción"].str.contains("comprada")].iloc[0]
    d = f["σ GARCH"] / f["Ask"] - 1
    lin.append(f"**{t} {c['compra']} {c['K']:,.0f}**: con σ GARCH ({pc(c['vol_garch'])}) el árbol la valúa en **{f['σ GARCH']:.2f}** y el mercado pide **{f['Ask']:.2f}** "
               f"({d:+.1%}). " + ("GARCH la ve **más cara** de lo que vale: se paga de más." if d < -0.05 else
                                  "GARCH la ve **más barata**: hay descuento frente a lo que pide el mercado." if d > 0.05 else "El precio es **justo** según GARCH."))
dif_iv = (precios_op["σ implícita"] - precios_op["Mercado (medio)"]).abs(); pdif_iv = float((dif_iv / precios_op["Mercado (medio)"]).max())
leer(lin + [f"Con la σ implícita el árbol reproduce el precio medio de mercado con una diferencia máxima de {dif_iv.max():.2f} USD por acción ({pdif_iv:.1%}). No es exacta porque la volatilidad implícita que muestra TradingView sale de su propio modelo y las cotizaciones son las del último cierre; aun así sirve de comprobación de que el árbol está bien armado. La diferencia con la columna «σ GARCH» es justo el efecto de usar el pronóstico propio.",
            "Más volatilidad esperada = opción más cara. Como las dos opciones (call y put) suben o bajan de precio juntas con la volatilidad, esta comparación sirve para ambas."], "opciones")

fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, t in zip(axs, TICKERS):
    c = OPC[t]; f = precios_op[(precios_op["Acción"] == t) & precios_op["Opción"].str.contains("comprada")].iloc[0]
    etiquetas = ["σ histórica\n1 año", "σ GARCH", "σ implícita", "Mercado\n(medio)"]
    vals = [f["σ histórica 1a"], f["σ GARCH"], f["σ implícita"], f["Mercado (medio)"]]
    barras = ax.bar(etiquetas, vals, color=COLORES[t], alpha=0.9, width=0.62)
    barras[1].set_alpha(1.0); barras[0].set_alpha(0.45); barras[2].set_alpha(0.65); barras[3].set_facecolor("none"); barras[3].set_edgecolor(COLORES[t]); barras[3].set_linewidth(2)
    for b, v in zip(barras, vals):
        ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=INK)
    ax.set_title(f"{t} {c['compra']} {c['K']:,.0f}: valor por acción (USD)"); ax.set_ylim(0, max(vals) * 1.18)
mostrar()
leer("Cada barra es el valor de la opción que compramos según una σ distinta; la barra con contorno es lo que dice el mercado. Si la barra de GARCH queda más alta que la de mercado, la opción está barata.", "graf_opciones")

### 10b. Griegas con la σ de GARCH y con la σ implícita (por contrato de 100 acciones)
Delta: cuánto cambia el valor si la acción sube 1 USD. Gamma: cuánto cambia la delta. Theta: cuánto se pierde por cada día que pasa.
Vega: cuánto cambia si la volatilidad sube 1 punto. Rho: cuánto cambia si la tasa sube 1 punto.

In [ ]:
fg = []
GR = {}
for t in TICKERS:
    c = OPC[t]
    for nom, sg in (("σ GARCH", c["vol_garch"]), ("σ implícita", c["iv"])):
        g = griegas(c["S0"], c["K"], R_FREE, sg, c["T"], 500, c["compra"], True, c["q"])
        GR[(t, nom)] = g
        fg.append({"Acción": t, "Opción": f"{c['compra']} {c['K']:,.0f}", "σ usada": nom, "Valor (USD/acción)": g["precio"],
                   "Delta (acciones)": g["delta"] * MULT, "Gamma": g["gamma"] * MULT, "Theta (USD/día)": g["theta"] * MULT,
                   "Vega (USD por +1 pto)": g["vega"] * MULT, "Rho (USD por +1 pto)": g["rho"] * MULT})
gr_df = pd.DataFrame(fg)
tabla(gr_df, {"Valor (USD/acción)": "{:,.2f}", "Delta (acciones)": "{:+.1f}", "Gamma": "{:.2f}", "Theta (USD/día)": "{:+.2f}",
              "Vega (USD por +1 pto)": "{:+.2f}", "Rho (USD por +1 pto)": "{:+.2f}"}, "griegas")
lin = []
for t in TICKERS:
    g = GR[(t, "σ GARCH")]; c = OPC[t]
    lin.append(f"**{t} {c['compra']} {c['K']:,.0f}** (por contrato): equivale a {abs(g['delta']) * MULT:.0f} acciones; pierde **{abs(g['theta']) * MULT:.2f} USD por día** por el paso del tiempo; "
               f"si la volatilidad sube 1 punto gana {g['vega'] * MULT:.2f} USD.")
leer(lin + ["Theta es el costo de esperar: la opción compradora pierde valor cada día aunque la acción no se mueva. Vega es la apuesta a la volatilidad: por eso GARCH importa."], "griegas")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (clave, titulo) in zip(axs, (("Gamma", "Gamma (por contrato)"), ("Vega (USD por +1 pto)", "Vega (USD por +1 pto de volatilidad)"))):
    x = np.arange(len(TICKERS)); w = 0.36
    for k, (nom, alpha) in enumerate((("σ GARCH", 1.0), ("σ implícita", 0.45))):
        vals = [gr_df[(gr_df["Acción"] == t) & (gr_df["σ usada"] == nom)][clave].iloc[0] for t in TICKERS]
        barras = ax.bar(x + (k - 0.5) * w, vals, w, color=[COLORES[t] for t in TICKERS], alpha=alpha, label=nom)
        for b, v in zip(barras, vals):
            ax.annotate(f"{v:.1f}", (b.get_x() + b.get_width() / 2, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8, color=INK)
    ax.set_xticks(x); ax.set_xticklabels(TICKERS); ax.set_title(titulo)
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=INK2, alpha=a) for a in (1.0, 0.45)], labels=["σ GARCH (barra fuerte)", "σ implícita (barra clara)"], fontsize=8)
mostrar()

### 10c. Niveles de nodos del árbol (4 pasos, con la σ de GARCH)
Para ver el árbol «a mano» se dibujan solo 4 pasos (el cálculo real usa 500). En cada nodo se compara **ejercer ya** con **esperar**.

In [ ]:
def arbol_niveles(S0_, K, r, sigma, T, N=4, tipo="call", q=0.0):
    dt = T / N; u_ = np.exp(sigma * np.sqrt(dt)); d_ = 1 / u_
    p = (np.exp((r - q) * dt) - d_) / (u_ - d_); disc = np.exp(-r * dt); sg = 1 if tipo == "call" else -1
    S = [[S0_ * u_ ** (i - j) * d_ ** j for j in range(i + 1)] for i in range(N + 1)]
    V = [None] * (N + 1); EX = [None] * (N + 1)
    V[N] = [max(sg * (s - K), 0) for s in S[N]]; EX[N] = [False] * (N + 1)
    for i in range(N - 1, -1, -1):
        V[i], EX[i] = [], []
        for j in range(i + 1):
            cont = disc * (p * V[i + 1][j] + (1 - p) * V[i + 1][j + 1]); ex = max(sg * (S[i][j] - K), 0)
            V[i].append(max(cont, ex)); EX[i].append(ex > cont + 1e-12 and ex > 0)
    return S, V, EX, u_, d_, p

NODOS = {}
for t in TICKERS:
    c = OPC[t]
    S, V, EX, u_, d_, p = arbol_niveles(c["S0"], c["K"], R_FREE, c["vol_garch"], c["T"], 4, c["compra"], c["q"])
    NODOS[t] = dict(S=S, V=V, EX=EX, u=u_, d=d_, p=p)
    filas = [{"Paso": i, "Nodo (bajas)": j, "Precio de la acción": S[i][j], "Valor de la opción": V[i][j],
              "Mejor acción": "Ejercer" if EX[i][j] else ("Vence" if i == 4 else "Esperar")}
             for i in range(0, 5) for j in range(i + 1) if i <= 2 or i == 4]
    display(Markdown(f"**{t} · {c['compra']} {c['K']:,.0f} · σ GARCH {pc(c['vol_garch'])} · u = {u_:.4f}, d = {d_:.4f}, p = {p:.4f}** (pasos 0, 1, 2 y 4)"))
    tabla(pd.DataFrame(filas), {"Precio de la acción": "{:,.2f}", "Valor de la opción": "{:,.2f}"}, f"nodos_{t}")
leer([f"Cada paso son ~{DIAS_CAL / 4:.0f} días. En cada nodo la acción sube (×u) o baja (×d); el valor de la opción se calcula de atrás hacia adelante, igual que en el trabajo de árboles.",
      "«Ejercer» aparece cuando cobrar el valor intrínseco ya vale más que esperar; en una call sin dividendos casi nunca ocurre, y en una put sí puede ocurrir en los nodos donde la acción cayó mucho."] +
     [f"**{t}**: nodos de ejercicio anticipado en el árbol de 4 pasos: {sum(sum(r) for r in NODOS[t]['EX'])}." for t in TICKERS], "nodos")

## 11. Opciones americanas: ¿cuánto vale poder ejercer antes?
La diferencia entre la americana y la europea es la **prima de ejercicio anticipado**. Se calcula con la σ de GARCH.
- **Put**: puede convenir ejercer si la acción cae mucho (se cobra el strike y se ganan intereses).
- **Call**: solo conviene ejercer antes si la acción paga dividendos (XOM y, muy poco, NVDA).

In [ ]:
AMER = {}
filas = []
for t in TICKERS:
    c = OPC[t]
    for tipo in ("call", "put"):
        am = precio(c["S0"], c["K"], c["vol_garch"], c["T"], tipo, True, c["q"])
        eu = precio(c["S0"], c["K"], c["vol_garch"], c["T"], tipo, False, c["q"])
        filas.append({"Acción": t, "Opción": tipo, "Americana": am, "Europea": eu, "Prima de ejercicio anticipado": am - eu, "% del precio": 100 * (am - eu) / eu})
        AMER[(t, tipo)] = am - eu
    grid = np.linspace(0.3 * c["K"], c["K"], 60)
    ejercer = [precio(s, c["K"], c["vol_garch"], c["T"], "put", True, c["q"], N=300) - (c["K"] - s) < 1e-6 for s in grid]
    AMER[(t, "critico")] = float(grid[np.where(ejercer)[0].max()]) if any(ejercer) else None
amer_df = pd.DataFrame(filas)
tabla(amer_df, {"Americana": "{:,.3f}", "Europea": "{:,.3f}", "Prima de ejercicio anticipado": "{:,.3f}", "% del precio": "{:.2f}"}, "americanas")
lin = []
for t in TICKERS:
    c = OPC[t]; crit = AMER[(t, "critico")]
    lin.append(f"**{t}**: poder ejercer la put antes vale {AMER[(t, 'put')]:.2f} USD por acción. "
               + (f"Si cae por debajo de ~{crit:,.0f} (hoy {c['S0']:,.0f}), ejercerla hoy sería lo óptimo. " if crit else "En el rango razonable de precios no conviene ejercerla hoy: esperar vale más. ")
               + ("Sin dividendos, la call americana vale lo mismo que la europea." if c["q"] <= 0 else
                  f"La call americana vale {AMER[(t, 'call')]:.3f} más que la europea: el dividendo ({pc(c['q'], 2)}) es menor que la tasa libre de riesgo ({pc(R_FREE, 2)}), así que esperar sigue siendo mejor que ejercer." if AMER[(t, 'call')] < 0.005 else
                  f"La call americana vale {AMER[(t, 'call')]:.3f} más que la europea porque el dividendo ({pc(c['q'], 2)}) hace que ejercer antes tenga algo de valor."))
leer(lin + ["Regla práctica: **GARCH decide qué volatilidad usar; el árbol decide cuándo conviene ejercer** y cuánto vale la opción."], "americanas")

## 12. Comprobación: Monte Carlo GARCH (Duan) vs. árbol (opciones europeas)
Si el árbol con la volatilidad promedio de GARCH se parece al Monte Carlo con GARCH completo (volatilidad cambiando cada día),
la aproximación de Hull (23.6) es razonable para valorar la opción.

In [ ]:
rng = np.random.default_rng(SEED + 1)
comp = []
for t in TICKERS:
    c = OPC[t]
    S_rn = simular_trayectorias(t, c["S0"], H, N_SIM * 2, R_FREE - c["q"], rng)[-1]
    disc = np.exp(-R_FREE * c["T"])
    mc_c = disc * np.maximum(S_rn - c["K"], 0).mean(); mc_p = disc * np.maximum(c["K"] - S_rn, 0).mean()
    ar_c = precio(c["S0"], c["K"], c["vol_garch"], c["T"], "call", False, c["q"]); ar_p = precio(c["S0"], c["K"], c["vol_garch"], c["T"], "put", False, c["q"])
    comp.append({"Acción": t, "Call MC-GARCH": mc_c, "Call árbol (σ GARCH)": ar_c, "Dif. call %": 100 * (ar_c / mc_c - 1),
                 "Put MC-GARCH": mc_p, "Put árbol (σ GARCH)": ar_p, "Dif. put %": 100 * (ar_p / mc_p - 1)})
comp_df = pd.DataFrame(comp)
tabla(comp_df, _fmt({c: "{:,.3f}" for c in comp_df.columns if c != "Acción" and "%" not in c}, {"Dif. call %": "{:+.1f}", "Dif. put %": "{:+.1f}"}), "comp_mc")
dmax = max(comp_df["Dif. call %"].abs().max(), comp_df["Dif. put %"].abs().max())
leer([f"La diferencia más grande entre el árbol y el Monte Carlo completo es de {dmax:.1f} %. "
      + ("Es pequeña: usar la volatilidad promedio de GARCH dentro del árbol es una buena aproximación." if dmax < 5 else
         "Es notable: conviene tomar con cautela el precio del árbol y apoyarse también en el Monte Carlo."),
      "Para este ejercicio de 2 meses la volatilidad casi no cambia de nivel, por eso las dos formas coinciden."], "comp_mc")

## 12b. Validación contra los ejemplos de la presentación
Se reproducen los números de la presentación del profesor (TSLA y AAPL con parámetros supuestos) con **este mismo código**.
Si coinciden, las funciones de pronóstico y del árbol están bien implementadas.

In [ ]:
def prom_ejemplo(vol_hoy, vol_lp, ab, dias):
    VL = vol_lp ** 2 / 252 * 1e4; v0 = vol_hoy ** 2 / 252 * 1e4; k = np.arange(dias)
    return float(np.sqrt(252 * (VL + ab ** k * (v0 - VL)).mean() / 1e4))

filas = []
for nom, S, vh, vl, q, esp_vol, esp_put, esp_call in (("TSLA", 310, .65, .55, 0.0, .589, 48.10, 53.69), ("AAPL", 322, .20, .28, 0.004, .254, 20.52, 25.73)):
    sg = prom_ejemplo(vh, vl, 0.98, 126)
    pu = precio(S, S, sg, 0.5, "put", True, q, 500, 0.04); ca = precio(S, S, sg, 0.5, "call", True, q, 500, 0.04)
    filas += [{"Prueba": f"{nom}: σ promedio a 126 días", "Este código": sg, "Presentación": esp_vol},
              {"Prueba": f"{nom}: put americana ATM a 6 meses (σ GARCH)", "Este código": pu, "Presentación": esp_put},
              {"Prueba": f"{nom}: call americana ATM a 6 meses (σ GARCH)", "Este código": ca, "Presentación": esp_call}]
filas.append({"Prueba": "Hull cap. 13: put americana 50/52, 5 %, 30 %, 2 años, 2 pasos", "Este código": precio(50, 52, 0.30, 2, "put", True, 0, 2, 0.05), "Presentación": 7.4284})
val_df = pd.DataFrame(filas)
val_df["Diferencia"] = val_df["Este código"] - val_df["Presentación"]
tabla(val_df, {"Este código": "{:,.4f}", "Presentación": "{:,.4f}", "Diferencia": "{:+.4f}"}, "validacion")
leer(["Las volatilidades promedio coinciden con la presentación (58.9 % y 25.4 %) y los precios del árbol coinciden al centavo. "
      "La pequeña diferencia en la fila de volatilidad es solo el redondeo de la diapositiva.",
      "El árbol además reproduce el ejemplo del libro de Hull (7.4284), así que la mecánica de comparar «ejercer vs. esperar» está bien."], "validacion")

## 13. Estrategia: señal de volatilidad y resultado esperado
1. **Señal** (presentación, diapositiva 15): razón = σ GARCH promedio ÷ σ implícita.
   **> 1.10**: GARCH ve más volatilidad que el mercado → opción barata → **comprar volatilidad** (straddle largo: call + put).
   **0.90 – 1.10**: coinciden → **sin ventaja** en volatilidad. **< 0.90**: opción cara → vender volatilidad solo con cobertura y límites (riesgo alto).
2. **Resultado esperado**: con las trayectorias GARCH «del mundo real» se estima la ganancia o pérdida al vencimiento.
   Se compra al **ask** (precio al que realmente se ejecuta una compra).

In [ ]:
def senal(ratio):
    if ratio > 1 + UMBRAL_SENAL:
        return "COMPRAR volatilidad", "▲", "GARCH ve más volatilidad que el mercado: la opción está barata."
    if ratio < 1 - UMBRAL_SENAL:
        return "NO COMPRAR (opción cara)", "▼", "GARCH ve menos volatilidad que el mercado: la opción está cara."
    return "SIN VENTAJA en volatilidad", "●", "GARCH y mercado coinciden."

rng = np.random.default_rng(SEED + 2)
SENAL, POS = {}, {}
for t in TICKERS:
    c, m = OPC[t], MERCADO[t]
    ratio = c["vol_garch"] / c["iv"]
    etiqueta, icono, expl = senal(ratio)
    ST = simular_trayectorias(t, c["S0"], H, N_SIM * 2, MU_ANUAL, rng)[-1]
    ask_c, ask_p = m["call"][1], m["put"][1]
    costo_s = ask_c + ask_p
    pnl_s = np.maximum(ST - c["K"], 0) + np.maximum(c["K"] - ST, 0) - costo_s
    SENAL[t] = dict(ratio=ratio, etiqueta=etiqueta, icono=icono, expl=expl, costo=costo_s, be_inf=c["K"] - costo_s, be_sup=c["K"] + costo_s,
                    pnl_esp=pnl_s.mean(), prob_gan=(pnl_s > 0).mean(), ST=ST)
    # posición direccional (la opción de la tarea de árboles)
    tipo = c["compra"]; ask = m[tipo][1]; n = c["contratos"]
    pay = np.maximum(ST - c["K"], 0) if tipo == "call" else np.maximum(c["K"] - ST, 0)
    pnl_usd = (pay - ask) * MULT * n
    POS[t] = dict(tipo=tipo, ask=ask, n=n, costo=ask * MULT * n, be=c["K"] + ask if tipo == "call" else c["K"] - ask,
                  prob_itm=(pay > 0).mean(), prob_gan=(pnl_usd > 0).mean(), pnl_esp=pnl_usd.mean(),
                  p5=np.percentile(pnl_usd, 5), p95=np.percentile(pnl_usd, 95), pnl=pnl_usd)

# --- gráfica del straddle: payoff arriba, distribución GARCH abajo (un solo eje cada uno)
fig, axs = plt.subplots(2, 3, figsize=(14, 5.4), gridspec_kw={"height_ratios": [3, 1.3]})
for j, t in enumerate(TICKERS):
    c = OPC[t]; s = SENAL[t]; ST = s["ST"]
    grid = np.linspace(np.percentile(ST, 0.5), np.percentile(ST, 99.5), 200)
    a1, a2 = axs[0, j], axs[1, j]
    a1.plot(grid, np.maximum(grid - c["K"], 0) + np.maximum(c["K"] - grid, 0) - s["costo"], color=COLORES[t], lw=2.2)
    a1.axhline(0, color=INK2, lw=0.9)
    a1.axvline(c["S0"], color=INK2, ls=":", lw=1)
    for be in (s["be_inf"], s["be_sup"]):
        a1.axvline(be, color=INK, ls="--", lw=0.9)
    a1.set_title(f"{t}: resultado de un straddle largo (strike {c['K']:,.0f})"); a1.set_xlim(grid[0], grid[-1]); a1.set_ylabel("USD por acción")
    a1.annotate(f"equilibrio {s['be_inf']:,.0f} y {s['be_sup']:,.0f}", (0.02, 0.06), xycoords="axes fraction", fontsize=8, color=INK)
    a2.hist(ST, bins=60, range=(grid[0], grid[-1]), color=COLORES[t], alpha=0.5, density=True); a2.set_yticks([])
    a2.set_xlim(grid[0], grid[-1]); a2.set_xlabel("Precio al vencimiento (distribución GARCH)")
mostrar()

res_s = pd.DataFrame([{"Acción": t, "Razón GARCH / implícita": SENAL[t]["ratio"], "Señal": f"{SENAL[t]['icono']} {SENAL[t]['etiqueta']}",
                       "Costo del straddle (USD/acción)": SENAL[t]["costo"], "Equilibrio inferior": SENAL[t]["be_inf"], "Equilibrio superior": SENAL[t]["be_sup"],
                       "P&L esperado (USD/acción)": SENAL[t]["pnl_esp"], "Prob. de ganar": SENAL[t]["prob_gan"]} for t in TICKERS])
tabla(res_s, {"Razón GARCH / implícita": "{:.2f}", "Costo del straddle (USD/acción)": "{:,.2f}", "Equilibrio inferior": "{:,.2f}", "Equilibrio superior": "{:,.2f}",
              "P&L esperado (USD/acción)": "{:+,.2f}", "Prob. de ganar": "{:.1%}"}, "straddle")
lin = []
for t in TICKERS:
    s = SENAL[t]; c = OPC[t]
    lin.append(f"**{t}** · razón {s['ratio']:.2f} → **{s['etiqueta']}**. {s['expl']} Un straddle cuesta {s['costo']:.2f} por acción: gana solo si al vencimiento "
               f"la acción queda debajo de {s['be_inf']:,.0f} o arriba de {s['be_sup']:,.0f}. Probabilidad de ganar según GARCH: {pc(s['prob_gan'], 0)}.")
leer(lin + ["El straddle gana con movimientos grandes en cualquier dirección y pierde si la acción casi no se mueve. Por eso solo conviene cuando GARCH espera más volatilidad de la que cobra el mercado."], "straddle")

# --- posición direccional (la de la tarea de árboles)
display(Markdown("**Nuestra posición direccional (la opción comprada en la tarea de árboles), resultado al 18-dic según GARCH:**"))
pos_df = pd.DataFrame([{"Acción": t, "Opción": f"{POS[t]['tipo']} {OPC[t]['K']:,.0f}", "Contratos": POS[t]["n"], "Ask": POS[t]["ask"], "Costo total (USD)": POS[t]["costo"],
                        "Equilibrio": POS[t]["be"], "Prob. de terminar en el dinero": POS[t]["prob_itm"], "Prob. de ganar": POS[t]["prob_gan"],
                        "P&L esperado (USD)": POS[t]["pnl_esp"], "P5 (USD)": POS[t]["p5"], "P95 (USD)": POS[t]["p95"]} for t in TICKERS])
tabla(pos_df, {"Ask": "{:,.2f}", "Costo total (USD)": "{:,.0f}", "Equilibrio": "{:,.2f}", "Prob. de terminar en el dinero": "{:.1%}", "Prob. de ganar": "{:.1%}",
               "P&L esperado (USD)": "{:+,.0f}", "P5 (USD)": "{:+,.0f}", "P95 (USD)": "{:+,.0f}"}, "posicion")
fig, axs = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, t in zip(axs, TICKERS):
    p = POS[t]
    ax.hist(p["pnl"], bins=60, color=COLORES[t], alpha=0.85)
    ax.axvline(0, color=INK, lw=1); ax.axvline(-p["costo"], color=INK2, ls="--", lw=1)
    ax.annotate(f"pérdida máx. {-p['costo']:,.0f}", (-p["costo"], ax.get_ylim()[1] * 0.92), xytext=(4, 0), textcoords="offset points", fontsize=8, color=INK)
    ax.set_title(f"{t}: ganancia o pérdida (USD, {p['n']} contrato(s))"); ax.set_yticks([])
mostrar()
leer([f"**{t}** ({POS[t]['tipo']} {OPC[t]['K']:,.0f} × {POS[t]['n']}): costo {POS[t]['costo']:,.0f} USD, que es lo máximo que se puede perder. "
      f"Probabilidad de ganar {pc(POS[t]['prob_gan'], 0)}; resultado esperado {POS[t]['pnl_esp']:+,.0f} USD." for t in TICKERS] +
     ["Las barras muestran los resultados simulados: la columna de la izquierda (pérdida máxima) son los escenarios en que la opción vence sin valor. "
      f"El centro depende del rendimiento supuesto ({MU_ANUAL:.0%} anual), así que estas cifras son una guía, no una promesa."], "posicion")

## 13b. ¿Qué tan sólida es la señal?
La señal depende de **un** pronóstico. Antes de decidir se repite la razón (σ ÷ σ implícita) con otras formas de medir la volatilidad:
GARCH estimado con 5, 3 y 2 años de historia, la volatilidad histórica de 1 año y EWMA de hoy.
Si la mayoría coincide con la señal base (GARCH de 5 años), la señal es **sólida**; si no, es **frágil**.

In [ ]:
def garch_vol_ventana(t, n_obs):
    u = M[t]["u"][-n_obs:]; pr = ajustar_garch(u); v = var_garch(pr, u); omega, alpha, beta = pr
    VL = omega / (1 - alpha - beta); v_next = omega + alpha * u[-1] ** 2 + beta * v[-1]
    fv = VL + (alpha + beta) ** np.arange(H) * (v_next - VL)
    return float(np.sqrt(252 * fv.mean()) / 100), float(vol_anual_desde_var(VL))

ROB = {}
for t in TICKERS:
    g3, vl3 = garch_vol_ventana(t, 756); g2, vl2 = garch_vol_ventana(t, 504)
    med = {"GARCH 5 años": VOL_GARCH[t], "GARCH 3 años": g3, "GARCH 2 años": g2, "Histórica 1 año": float(vol_hist_1a[t]), "EWMA hoy": ewma_hoy[t]}
    razones = {k: v / IV[t] for k, v in med.items()}
    base_sen = senal(razones["GARCH 5 años"])[0]
    coinc = sum(senal(r)[0] == base_sen for k, r in razones.items() if k != "GARCH 5 años")
    ROB[t] = dict(razones=razones, coinc=coinc, solida=coinc >= 3, vl3=vl3, vl2=vl2, senal=base_sen)
rob_df = pd.DataFrame([{"Acción": t, **ROB[t]["razones"], "Coinciden con la señal base (de 4)": ROB[t]["coinc"],
                        "Solidez": "Sólida" if ROB[t]["solida"] else "Frágil"} for t in TICKERS])
tabla(rob_df, {k: "{:.2f}" for k in ROB[TICKERS[0]]["razones"]}, "robustez")
lin = []
for t in TICKERS:
    r = ROB[t]["razones"]
    lin.append(f"**{t}**: razón {r['GARCH 5 años']:.2f} con GARCH de 5 años; {r['GARCH 3 años']:.2f} con 3 años; {r['GARCH 2 años']:.2f} con 2 años; "
               f"{r['Histórica 1 año']:.2f} con la histórica de 1 año y {r['EWMA hoy']:.2f} con EWMA de hoy. Coinciden {ROB[t]['coinc']} de 4 → **señal {'sólida' if ROB[t]['solida'] else 'frágil'}**. "
               f"La volatilidad «normal» a la que regresa GARCH es {pc(VOL_LP[t])} con 5 años, {pc(ROB[t]['vl3'])} con 3 y {pc(ROB[t]['vl2'])} con 2.")
leer(lin + ["Razón mayor a 1.10 = opción barata; menor a 0.90 = cara; en medio = sin ventaja. Una señal frágil significa que depende de cómo se estime la volatilidad: "
            "por ejemplo, de que GARCH regrese a un nivel «normal» calculado con años en que la acción se comportó distinto.",
            "Una razón apenas arriba de 1.10 o abajo de 0.90 también es débil: un cambio pequeño en los datos la mueve de categoría."], "robustez")

## 13c. Decisión para Investopedia
Se aplica la regla de la señal a cada acción. El tamaño usa el mismo presupuesto por posición que el trabajo de árboles
(`PRESUPUESTO_POSICION`). **Costo total = ask × 100 × contratos.** La pérdida máxima de una opción comprada es lo que se paga.

In [ ]:
PRESUPUESTO_POSICION = 3000      # USD por posición (el mismo del trabajo de árboles binomiales)
CAPITAL = 100000                 # capital del portafolio en Investopedia (USD)

def contratos_para(costo_unitario):
    return max(1, int(round(PRESUPUESTO_POSICION / (costo_unitario * MULT))))

DEC = {}
filas = []
for t in TICKERS:
    c, m, s, p = OPC[t], MERCADO[t], SENAL[t], POS[t]
    if s["ratio"] > 1 + UMBRAL_SENAL:
        accion = "COMPRAR straddle (call + put)"
        n = contratos_para(s["costo"])
        detalle = f"{t} 18-dic-2026 · CALL {c['K']:,.0f} @ {m['call'][1]:.2f} + PUT {c['K']:,.0f} @ {m['put'][1]:.2f}"
        costo, pmax, be = s["costo"] * MULT * n, s["costo"] * MULT * n, f"{s['be_inf']:,.0f} / {s['be_sup']:,.0f}"
        pg, pe = s["prob_gan"], s["pnl_esp"] * MULT * n
        motivo = "GARCH espera más volatilidad que la que cobra el mercado."
        if not ROB[t]["solida"]:
            accion += " (señal frágil)"
            motivo += " Ojo: la señal es frágil, otras medidas de volatilidad no la confirman (ver sección 13b)."
    elif s["ratio"] < 1 - UMBRAL_SENAL:
        accion = "NO COMPRAR por volatilidad (opción cara)"
        n = 0; detalle = f"Si se compra de todos modos por tesis direccional: {t} {p['tipo'].upper()} {c['K']:,.0f} @ {p['ask']:.2f}"
        costo = pmax = 0.0; be = "—"; pg, pe = p["prob_gan"], p["pnl_esp"]
        motivo = "GARCH espera menos volatilidad que el mercado: se estaría pagando de más. Ojo: GARCH no ve el reporte de resultados que sí está en la implícita."
        if not ROB[t]["solida"]:
            motivo += " La señal es frágil (ver sección 13b)."
    else:
        accion = f"MANTENER solo la {p['tipo'].upper()} direccional (sin ventaja de volatilidad)"
        n = contratos_para(p["ask"]); detalle = f"{t} 18-dic-2026 · {p['tipo'].upper()} {c['K']:,.0f} @ {p['ask']:.2f}"
        costo = pmax = p["ask"] * MULT * n; be = f"{p['be']:,.2f}"
        pn = (np.maximum(SENAL[t]["ST"] - c["K"], 0) if p["tipo"] == "call" else np.maximum(c["K"] - SENAL[t]["ST"], 0)) - p["ask"]
        pg, pe = (pn > 0).mean(), pn.mean() * MULT * n
        motivo = "El precio de la opción es justo según GARCH; solo se justifica si hay una tesis direccional (la de la tarea de árboles)."
        if pe < 0:
            motivo += f" Ojo: con el rendimiento supuesto ({MU_ANUAL:.0%} anual) el resultado esperado es negativo ({pe:+,.0f} USD) y la probabilidad de ganar es {pg:.0%}: es una apuesta direccional, no una ventaja de volatilidad."
    if n and costo > 1.5 * PRESUPUESTO_POSICION:
        motivo += f" Ojo: un solo contrato ya cuesta {costo:,.0f} USD, más que el presupuesto por posición ({PRESUPUESTO_POSICION:,.0f} USD); decidir si se justifica ese tamaño."
    DEC[t] = dict(accion=accion, n=n, detalle=detalle, costo=costo, pmax=pmax, be=be, prob_gan=pg, pnl_esp=pe, motivo=motivo, ratio=s["ratio"])
    filas.append({"Acción": t, "Razón GARCH/implícita": s["ratio"], "Decisión": accion, "Contrato": detalle, "Contratos": n,
                  "Costo total (USD)": costo, "Pérdida máxima (USD)": pmax, "Equilibrio": be, "Prob. de ganar": pg, "P&L esperado (USD)": pe})
dec_df = pd.DataFrame(filas)
tabla(dec_df, {"Razón GARCH/implícita": "{:.2f}", "Costo total (USD)": "{:,.0f}", "Pérdida máxima (USD)": "{:,.0f}", "Prob. de ganar": "{:.0%}", "P&L esperado (USD)": "{:+,.0f}"}, "decision")
total_dec = sum(DEC[t]["costo"] for t in TICKERS)
leer([f"**{t}**: {DEC[t]['accion']}. {DEC[t]['motivo']}" for t in TICKERS] +
     [f"Costo total de las decisiones: **{total_dec:,.0f} USD**, es decir {pc(total_dec / CAPITAL)} del capital de {CAPITAL:,.0f} USD (la pérdida máxima es ese monto)."] +
     ["Esta tabla es la que se lleva a Investopedia: contrato, cantidad y costo. Si la decisión es «no comprar» pero el equipo quiere conservar la posición direccional de la tarea de árboles, "
      "la fila muestra su costo, probabilidad de ganar y resultado esperado para justificarlo.",
      "Siempre hay riesgo: una opción comprada puede perder todo lo pagado. Ejercicio académico, no recomendación de inversión."], "decision")

## 13d. Rentabilidad a 2 semanas
La tarea pide la rentabilidad **en las próximas 2 semanas** (hasta la entrega del proyecto final), no solo al vencimiento.
Se toman las trayectorias GARCH a **10 días hábiles**, se **revalúa** cada opción con lo que le queda de vida (Black-Scholes con la volatilidad implícita de hoy, como aproximación)
y se supone que se **vende al bid** (se resta medio diferencial bid-ask). La ganancia se mide contra el **ask** pagado.

In [ ]:
from scipy.stats import norm
HORIZ_DIAS = 10                                          # días hábiles (≈ 2 semanas)
T_REST = max(T_ANIOS - HORIZ_DIAS * 7 / 5 / 365, 1 / 365)

def bs_vec(S, K, T_, r, sg, q, tipo):
    d1 = (np.log(S / K) + (r - q + 0.5 * sg ** 2) * T_) / (sg * np.sqrt(T_)); d2 = d1 - sg * np.sqrt(T_)
    if tipo == "call":
        return S * np.exp(-q * T_) * norm.cdf(d1) - K * np.exp(-r * T_) * norm.cdf(d2)
    return K * np.exp(-r * T_) * norm.cdf(-d2) - S * np.exp(-q * T_) * norm.cdf(-d1)

R2, filas = {}, []
for t in TICKERS:
    c, m = OPC[t], MERCADO[t]
    Sh = SIM[t][HORIZ_DIAS]
    val = {tp: bs_vec(Sh, c["K"], T_REST, R_FREE, c["iv"], c["q"], tp) - (m[tp][1] - m[tp][0]) / 2 for tp in ("call", "put")}
    posiciones = []
    if DEC[t]["n"] and DEC[t]["accion"].startswith("COMPRAR"):
        posiciones.append(("Straddle recomendado", DEC[t]["n"], m["call"][1] + m["put"][1], val["call"] + val["put"]))
    p_ = POS[t]
    posiciones.append((f"{p_['tipo'].upper()} {c['K']:,.0f} (tarea de árboles)", p_["n"], p_["ask"], val[p_["tipo"]]))
    for nom, n, costo_u, v in posiciones:
        pnl = (v - costo_u) * MULT * n
        R2[(t, nom)] = pnl
        filas.append({"Acción": t, "Posición": nom, "Contratos": n, "Costo (USD)": costo_u * MULT * n, "P&L esperado 2 sem. (USD)": pnl.mean(),
                      "Rentabilidad esperada": pnl.mean() / (costo_u * MULT * n), "Prob. de ganar": (pnl > 0).mean(),
                      "P5 (USD)": np.percentile(pnl, 5), "P95 (USD)": np.percentile(pnl, 95)})
r2_df = pd.DataFrame(filas)
tabla(r2_df, {"Costo (USD)": "{:,.0f}", "P&L esperado 2 sem. (USD)": "{:+,.0f}", "Rentabilidad esperada": "{:+.1%}", "Prob. de ganar": "{:.0%}",
              "P5 (USD)": "{:+,.0f}", "P95 (USD)": "{:+,.0f}"}, "rent2s")
leer([f"**{f['Acción']} · {f['Posición']}**: costo {f['Costo (USD)']:,.0f} USD; en 2 semanas se espera {f['P&L esperado 2 sem. (USD)']:+,.0f} USD ({f['Rentabilidad esperada']:+.1%}) "
      f"con probabilidad de ganar {pc(f['Prob. de ganar'], 0)} (rango P5–P95: {f['P5 (USD)']:+,.0f} a {f['P95 (USD)']:+,.0f})." for f in filas] +
     [f"Se revalúan las opciones a {HORIZ_DIAS} días hábiles con {T_REST * 365:.0f} días de vida restante. En 2 semanas la opción aún tiene valor temporal, por eso el resultado lo explica sobre todo "
      "el movimiento del precio (delta) y la caída por el paso del tiempo (theta).",
      "**Lectura importante:** a 2 semanas el resultado esperado queda cerca de cero. La ventaja de GARCH (opción barata o cara) se cobra cuando la volatilidad implícita del mercado se acerca a la que GARCH espera, y aquí se mantuvo constante; además se paga el diferencial bid-ask al salir. Por eso la ganancia en 2 semanas depende sobre todo de hacia dónde se mueva el precio, y la ventaja de volatilidad pesa más si se conserva hasta el vencimiento.",
      "Es una estimación con el rendimiento supuesto y la volatilidad implícita constante. La rentabilidad real se mostrará con las capturas de Investopedia."], "rent2s")

## 14. Resumen e interpretación

In [ ]:
RES = pd.DataFrame([{
    "Acción": t, "Vol. hoy": VOL_HOY[t], "Vol. largo plazo": VOL_LP[t], "σ GARCH a vencimiento": VOL_GARCH[t], "σ implícita": IV[t],
    "Razón": SENAL[t]["ratio"], "Señal": f"{SENAL[t]['icono']} {SENAL[t]['etiqueta']}", "Persistencia": M[t]["p"][1] + M[t]["p"][2],
    "Mediana a vencimiento": res_mc[t]["Mediana"], "P5": res_mc[t]["P5"], "P95": res_mc[t]["P95"]} for t in TICKERS])
tabla(RES, {"Vol. hoy": "{:.1%}", "Vol. largo plazo": "{:.1%}", "σ GARCH a vencimiento": "{:.1%}", "σ implícita": "{:.1%}", "Razón": "{:.2f}", "Persistencia": "{:.3f}",
            "Mediana a vencimiento": "{:,.0f}", "P5": "{:,.0f}", "P95": "{:,.0f}"}, "resumen")

resumen = [f"Fuente de datos: {fuente}. Cotizaciones al {FECHA_COTIZACION}; vencimiento {VENCIMIENTO} ({DIAS_CAL} días naturales, {H} hábiles)."]
for t in TICKERS:
    a, b = M[t]["p"][1], M[t]["p"][2]
    resumen.append(f"**{t}**: GARCH α={a:.3f}, β={b:.3f} (persistencia {a + b:.3f}). Volatilidad hoy {pc(VOL_HOY[t])} → normal {pc(VOL_LP[t])}; "
                   f"promedio esperado hasta el 18-dic **{pc(VOL_GARCH[t])}** vs. {pc(IV[t])} del mercado (razón {SENAL[t]['ratio']:.2f}). "
                   f"Rango de precio al vencimiento (P5–P95): {res_mc[t]['P5']:,.0f}–{res_mc[t]['P95']:,.0f}. **Decisión: {DEC[t]['accion']}.**")
resumen += ["**GARCH vs. árbol:** el árbol binomial valora la opción; GARCH le dice **qué volatilidad usar** durante su vida.",
            "**Limitaciones:** GARCH(1,1) supone normalidad, no captura asimetría (para eso existen GJR-GARCH y EGARCH), no conoce las fechas de reporte "
            "y los resultados dependen de los datos y de los supuestos (rendimiento esperado, tasa libre de riesgo, cotizaciones del 30-oct).",
            "Material educativo; no es una recomendación de inversión."]
leer(resumen, "resumen", "Conclusiones")

In [ ]:
print("Listo. (Esta versión para LSEG CodeBook no genera el dashboard HTML; el dashboard se entrega aparte.)")